# RandomForest Experiment — IEEE-CIS Fraud Detection

## 0. Setup & Imports

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'mlflow', 'dagshub', 'optuna', '--quiet'], capture_output=True)

import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import mlflow, mlflow.sklearn, dagshub, optuna
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import SelectFromModel
from sklearn.base import BaseEstimator, TransformerMixin
print('Ready!')

In [ ]:
DAGSHUB_USERNAME = 'YOUR_DAGSHUB_USERNAME'
DAGSHUB_REPO     = 'YOUR_REPO_NAME'
dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)
mlflow.set_experiment('RandomForest_Training')

## 1. Data Loading

In [ ]:
BASE = '/kaggle/input/ieee-fraud-detection/'
train = pd.read_csv(BASE+'train_transaction.csv').merge(pd.read_csv(BASE+'train_identity.csv'), on='TransactionID', how='left')
test  = pd.read_csv(BASE+'test_transaction.csv').merge(pd.read_csv(BASE+'test_identity.csv'),  on='TransactionID', how='left')
# Use a sample for RF (memory heavy)
# train = train.sample(frac=0.5, random_state=42)  # Uncomment if OOM
print(train.shape, test.shape)

## 2. Cleaning

In [ ]:
with mlflow.start_run(run_name='RandomForest_Cleaning'):
    drop_cols = train.isnull().mean()[lambda x: x > 0.9].index.tolist()
    train.drop(columns=drop_cols, inplace=True)
    test.drop(columns=[c for c in drop_cols if c in test], inplace=True)

    const_cols = [c for c in train.columns
                  if train[c].nunique(dropna=False) <= 1]
    train.drop(columns=const_cols, inplace=True)
    test.drop(columns=[c for c in const_cols if c in test], inplace=True)

    for col in ['P_emaildomain','R_emaildomain']:
        if col in train:
            top = train[col].value_counts().nlargest(10).index
            train[col] = train[col].where(train[col].isin(top), 'other')
            test[col]  = test[col].where(test[col].isin(top), 'other')

    mlflow.log_param('dropped_high_missing', len(drop_cols))
    mlflow.log_param('dropped_constant', len(const_cols))
    mlflow.log_metric('cols_remaining', train.shape[1])
    print(f'After cleaning: {train.shape}')

## 3. Feature Engineering

In [ ]:
with mlflow.start_run(run_name='RandomForest_Feature_Engineering'):

    def engineer(df):
        df = df.copy()
        df['hour']        = (df['TransactionDT'] / 3600) % 24
        df['day_of_week'] = (df['TransactionDT'] / (3600*24)) % 7
        df['is_night']    = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
        df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
        df['TransactionAmt_log']   = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_cents'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        if 'P_emaildomain' in df and 'R_emaildomain' in df:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        df['nan_count'] = df.isnull().sum(axis=1)
        return df

    train = engineer(train)
    test  = engineer(test)

    TARGET   = 'isFraud'
    DROP_COLS= ['TransactionID', 'TransactionDT', TARGET]

    cat_cols = [c for c in train.select_dtypes(include='object').columns if c not in DROP_COLS]
    for col in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([train[col], test[col]]).astype(str)
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col]  = le.transform(test[col].astype(str))

    feature_cols = [c for c in train.columns if c not in DROP_COLS]
    X = train[feature_cols].fillna(-999)
    y = train[TARGET]
    X_test = test[feature_cols].fillna(-999)

    mlflow.log_metric('features_after_fe', X.shape[1])
    print(f'Shape after FE: {X.shape}')

## 4. Feature Selection

In [ ]:
with mlflow.start_run(run_name='RandomForest_Feature_Selection'):
    # RF importance — quick model
    quick_rf = RandomForestClassifier(
        n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
    quick_rf.fit(X, y)

    importances = pd.Series(quick_rf.feature_importances_, index=feature_cols)
    top_features = importances.nlargest(100).index.tolist()

    fig, ax = plt.subplots(figsize=(10, 8))
    importances.nlargest(30).plot(kind='barh', ax=ax)
    ax.set_title('RandomForest Feature Importances')
    plt.tight_layout()
    plt.savefig('rf_importance.png')
    mlflow.log_artifact('rf_importance.png')

    mlflow.log_param('fs_method', 'rf_importance_top100')
    mlflow.log_metric('features_selected', len(top_features))

    X_sel      = X[top_features]
    X_test_sel = X_test[top_features]
    print(f'Selected {len(top_features)} features')

## 5. Training

### 5a. Underfitted — Very Shallow (max_depth=2)

In [ ]:
with mlflow.start_run(run_name='RF_Underfitted'):
    params_u = dict(n_estimators=20, max_depth=2, n_jobs=-1, random_state=42)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(RandomForestClassifier(**params_u), X_sel, y, cv=cv, scoring='roc_auc')
    mlflow.log_params(params_u)
    mlflow.log_metric('cv_auc_mean', scores.mean())
    mlflow.log_param('note', 'underfitted_shallow_few_trees')
    print(f'[UNDERFITTED] CV AUC: {scores.mean():.4f}')

### 5b. Overfitted — Very Deep (max_depth=None)

In [ ]:
with mlflow.start_run(run_name='RF_Overfitted'):
    params_o = dict(n_estimators=200, max_depth=None, min_samples_split=2,
                    min_samples_leaf=1, n_jobs=-1, random_state=42)
    m_o = RandomForestClassifier(**params_o)
    m_o.fit(X_sel, y)
    train_auc = roc_auc_score(y, m_o.predict_proba(X_sel)[:, 1])
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_auc = cross_val_score(RandomForestClassifier(**params_o), X_sel, y,
                              cv=cv, scoring='roc_auc').mean()
    mlflow.log_params(params_o)
    mlflow.log_metric('train_auc', train_auc)
    mlflow.log_metric('cv_auc', cv_auc)
    mlflow.log_metric('overfit_gap', train_auc - cv_auc)
    print(f'[OVERFITTED] Train: {train_auc:.4f} CV: {cv_auc:.4f} Gap: {train_auc-cv_auc:.4f}')
    print('Note: RF with no depth limit memorizes training data → overfit')

### 5c. Optuna Tuning

In [ ]:
def rf_objective(trial):
    params = {
        'n_estimators':       trial.suggest_int('n_estimators', 100, 500),
        'max_depth':          trial.suggest_int('max_depth', 5, 20),
        'min_samples_split':  trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':   trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features':       trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
        'n_jobs': -1, 'random_state': 42
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    return cross_val_score(RandomForestClassifier(**params), X_sel, y,
                           cv=cv, scoring='roc_auc').mean()

study = optuna.create_study(direction='maximize')
study.optimize(rf_objective, n_trials=20)
best_params = study.best_params
best_params.update({'n_jobs': -1, 'random_state': 42})
print(f'Best AUC: {study.best_value:.4f}')

### 5d. Final CV + Pipeline

In [ ]:
with mlflow.start_run(run_name='RF_Final_CV') as final_run:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    test_preds = np.zeros(len(X_test_sel))
    fold_aucs = []

    for fold, (tr_i, val_i) in enumerate(cv.split(X_sel, y)):
        m = RandomForestClassifier(**best_params)
        m.fit(X_sel.iloc[tr_i], y.iloc[tr_i])
        val_pred = m.predict_proba(X_sel.iloc[val_i])[:, 1]
        oof[val_i] = val_pred
        test_preds += m.predict_proba(X_test_sel)[:, 1] / 5
        fa = roc_auc_score(y.iloc[val_i], val_pred)
        fold_aucs.append(fa)
        print(f'  Fold {fold+1}: {fa:.4f}')

    oof_auc = roc_auc_score(y, oof)
    mlflow.log_params(best_params)
    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.log_metric('cv_auc_mean', np.mean(fold_aucs))
    print(f'OOF AUC: {oof_auc:.4f}')


class RFPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, selected_features=None):
        self.selected_features = selected_features
        self.label_encoders_ = {}
        self.cat_cols_ = []

    def fit(self, X, y=None):
        df = self._engineer(X.copy())
        self.cat_cols_ = df.select_dtypes(include='object').columns.tolist()
        for col in self.cat_cols_:
            le = LabelEncoder(); le.fit(df[col].astype(str))
            self.label_encoders_[col] = le
        return self

    def transform(self, X):
        df = self._engineer(X.copy())
        for col in self.cat_cols_:
            if col in df:
                le = self.label_encoders_[col]
                df[col] = df[col].astype(str).map(lambda x: x if x in le.classes_ else le.classes_[0])
                df[col] = le.transform(df[col])
        if self.selected_features:
            df = df[[f for f in self.selected_features if f in df.columns]]
        return df.fillna(-999)

    def _engineer(self, df):
        df['hour']        = (df['TransactionDT'] / 3600) % 24
        df['day_of_week'] = (df['TransactionDT'] / (3600*24)) % 7
        df['is_night']    = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
        df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
        df['TransactionAmt_log']   = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_cents'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        df['nan_count']   = df.isnull().sum(axis=1)
        if 'P_emaildomain' in df and 'R_emaildomain' in df:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        return df


X_raw = train.drop(columns=['isFraud','TransactionID'], errors='ignore')
y_raw = train['isFraud']

rf_pipeline = Pipeline([
    ('preprocessor', RFPreprocessor(selected_features=top_features)),
    ('classifier',   RandomForestClassifier(**best_params))
])
rf_pipeline.fit(X_raw, y_raw)

with mlflow.start_run(run_name='RF_Pipeline_Registry'):
    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.sklearn.log_model(
        sk_model=rf_pipeline,
        artifact_path='rf_fraud_pipeline',
        registered_model_name='RandomForest_Fraud_Pipeline'
    )
    print('RF pipeline registered!')

np.save('rf_test_preds.npy', test_preds)